# Семинар 09. Области видимости и декораторы

Функцию можно настроить один раз и сохранить получившееся поведение. А можно обернуть готовую функцию, добавив проверку, счётчик или журнал. Сегодня напишем пять таких инструментов и проверим, что настройки и состояние не перепутались между вызовами.

## Цели

- возвращать функцию, которая использует настройки внешнего вызова;
- отделять состояние одного замыкания от другого;
- обновлять сохранённое значение через `nonlocal`;
- сохранять аргументы, результат и исключения оборачиваемой функции;
- писать обычный декоратор и декоратор с параметром;
- сохранять имя и описание исходной функции через `functools.wraps`.

## Перед началом

Нужны функции, строки, списки, словари и исключения из первых восьми занятий, а также [лекция 9](lecture.ipynb). Классы, генераторы и асинхронный код не используются.

Задания 1–2 посвящены замыканиям, 3–4 — обёрткам, 5 — декоратору с параметром. В заготовках стоит `pass`. Сначала напишите нужную функцию и выполните её ячейку; затем выполните ячейку с определением проверки и снимите `#` перед вызовом `check_task_N()`.

`Run All` исходного ноутбука проходит, но это не означает, что задачи решены: вызовы проверок закомментированы. Готовых решений в ноутбуке нет.

`Callable` в аннотациях означает вызываемый объект; здесь это функция. Если условие не требует иного, не используйте глобальные переменные. Во всех задачах типы аргументов соответствуют условию.

In [ ]:
from collections.abc import Callable
from functools import wraps

## Задание 1. Проверка начала строки

Напишите `make_prefix_filter(prefix)`, которая возвращает функцию-предикат. Предикат получает строку и возвращает `True`, если она начинается с заданного префикса, иначе `False`.

Регистр и пробелы значимы. Пустой префикс подходит любой строке. Созданные предикаты должны работать независимо: создание новой проверки не меняет настройки старой.

```python
is_note = make_prefix_filter("note:")
assert is_note("note: купить хлеб") is True
assert is_note("Note: купить хлеб") is False
```

Длина строки и префикса — до 100 000 символов. Создание предиката не должно копировать строку-префикс. Время проверки — `O(min(len(prefix), len(text)))`, дополнительная память — `O(1)`.

In [ ]:
def make_prefix_filter(prefix: str) -> Callable[[str], bool]:
    pass

In [ ]:
def check_task_1():
    is_note = make_prefix_filter("note:")
    is_warning = make_prefix_filter("warning:")
    assert is_note("note: купить хлеб") is True
    assert is_note("Note: купить хлеб") is False
    assert is_note("warning: дождь") is False
    assert is_warning("warning: дождь") is True
    assert is_note("note:") is True
    assert is_note("") is False
    assert make_prefix_filter("")("") is True
    assert make_prefix_filter("")("любая строка") is True
    assert make_prefix_filter(" ё")(" ёлка") is True
    print("Задание 1: проверки пройдены")

# check_task_1()

## Задание 2. Независимые последовательности

Напишите `make_sequence(start, step)`, возвращающую функцию без аргументов. Первый вызов возвращает `start`, каждый следующий — предыдущее выданное значение плюс `step`.

Нулевой и отрицательный шаг разрешены. Каждая созданная последовательность хранит собственное состояние; нельзя использовать общий глобальный счётчик.

```python
next_number = make_sequence(10, 3)
assert [next_number(), next_number(), next_number()] == [10, 13, 16]
```

`start` и `step` — целые числа по модулю до 10⁹, число вызовов — до 100 000. Время и дополнительная память одного вызова — `O(1)` при обычной модели стоимости целочисленной арифметики.

In [ ]:
def make_sequence(start: int, step: int = 1) -> Callable[[], int]:
    pass

In [ ]:
def check_task_2():
    forward = make_sequence(10, 3)
    backward = make_sequence(2, -2)
    assert forward() == 10
    assert backward() == 2
    assert forward() == 13
    assert backward() == 0
    assert backward() == -2
    assert forward() == 16
    constant = make_sequence(0, 0)
    assert [constant(), constant(), constant()] == [0, 0, 0]
    default_step = make_sequence(-1)
    assert [default_step(), default_step(), default_step()] == [-1, 0, 1]
    print("Задание 2: проверки пройдены")

# check_task_2()

## Задание 3. Счётчик успешных вызовов

Напишите `make_counted(func)`. Она возвращает пару функций `(wrapped, get_count)`:

- `wrapped` принимает те же вызовы, что `func`, передаёт все позиционные и именованные аргументы и возвращает исходный результат;
- после успешного возврата `func` счётчик увеличивается на 1;
- если `func` поднимает исключение, оно выходит наружу без подмены, счётчик не меняется;
- `get_count()` возвращает текущее число успешных вызовов и ничего не изменяет.

У каждой пары свой счётчик, изначально 0. Сохраните имя и описание `func` через `wraps`. Ложные значения результата (`0`, `False`, `None`, пустой список) тоже означают успешный возврат.

Дополнительная работа обёртки на один вызов — `O(1)` без учёта передачи аргументов и работы `func`; число вызовов — до 100 000.

In [ ]:
def make_counted(func: Callable) -> tuple[Callable, Callable[[], int]]:
    pass

In [ ]:
def check_task_3():
    def divide(a, b=1):
        """Поделить a на b."""
        return a / b

    wrapped, get_count = make_counted(divide)
    other, other_count = make_counted(divide)
    assert get_count() == 0
    assert other_count() == 0
    assert wrapped(6, b=3) == 2.0
    assert get_count() == 1
    assert get_count() == 1
    assert wrapped(a=0, b=3) == 0.0
    assert get_count() == 2
    try:
        wrapped(1, b=0)
    except ZeroDivisionError:
        pass
    else:
        raise AssertionError("Ошибка оборачиваемой функции должна выйти наружу")
    assert get_count() == 2
    assert other(8, 2) == 4.0
    assert other_count() == 1
    assert get_count() == 2
    assert wrapped.__name__ == divide.__name__
    assert wrapped.__doc__ == divide.__doc__

    result = []
    original = ValueError("Ошибка внутри функции")
    def produce(*, fail=False):
        if fail:
            raise original
        return result

    produce_wrapped, produced = make_counted(produce)
    assert produce_wrapped() is result
    assert produced() == 1
    try:
        produce_wrapped(fail=True)
    except ValueError as error:
        assert error is original
    else:
        raise AssertionError("Ожидалось исходное исключение")
    assert produced() == 1
    print("Задание 3: проверки пройдены")

# check_task_3()

## Задание 4. Проверка до вызова

Напишите декоратор `reject_empty`. Оборачиваемая функция принимает строковый параметр `text` первым позиционным или по имени, затем может принимать другие аргументы.

Если `text` пуст или состоит из пробельных символов, поднимите `ValueError` **до вызова исходной функции**. Иначе передайте ей исходный текст без изменения, остальные аргументы и верните её результат. Ошибки из самой функции должны выходить наружу без подмены.

Сохраните имя и описание через `wraps`. Проверка не должна удалять пробелы из допустимой строки: `"  заметка  "` нужно передать в таком же виде.

Число вызовов — до 100 000, длина текста — до 100 000. Дополнительное время — `O(len(text))`; дополнительная память — не более `O(len(text))`.

In [ ]:
def reject_empty(func: Callable) -> Callable:
    pass

In [ ]:
def check_task_4():
    calls = []
    original = ValueError("Ошибка оформления")

    def render(text, prefix="", *, suffix=""):
        """Оформить текст."""
        calls.append(text)
        if text == "ошибка":
            raise original
        return f"{prefix}{text}{suffix}"

    checked = reject_empty(render)
    assert checked(" x ", "[", suffix="]") == "[ x ]"
    assert checked(text="заметка", suffix="!") == "заметка!"
    assert calls == [" x ", "заметка"]
    for text in ["", " ", "\n\t"]:
        try:
            checked(text)
        except ValueError:
            pass
        else:
            raise AssertionError("Пустой текст нужно отклонить")
    assert calls == [" x ", "заметка"]
    try:
        checked("ошибка")
    except ValueError as error:
        assert error is original
    else:
        raise AssertionError("Ошибка исходной функции была подавлена")
    assert calls[-1] == "ошибка"
    assert checked.__name__ == render.__name__
    assert checked.__doc__ == render.__doc__
    print("Задание 4: проверки пройдены")

# check_task_4()

## Задание 5. Общий журнал для разных функций

Напишите `journal(events)` — фабрику декораторов. Один созданный декоратор можно применить к нескольким функциям; они пишут в переданный общий список событий.

Каждый вызов обёртки добавляет пары `(имя_исходной_функции, событие)`:

1. Перед вызовом исходной функции — `"start"`.
2. При успешном возврате — `"ok"`.
3. При исключении, производном от `Exception`, — `"error"`, после чего исходное исключение передаётся дальше без подмены.

Для одного вызова после `"start"` будет ровно одно событие: `"ok"` или `"error"`. Прежние записи списка сохраняются. Возвращаемый результат, позиционные и именованные аргументы передаются без изменений. Сохраните имя и описание функции через `wraps`.

Проверка `KeyboardInterrupt` и других исключений вне группы `Exception` здесь не требуется. Число вызовов — до 100 000; дополнительная работа на вызов, кроме передачи аргументов и работы функции, — `O(1)`.

In [ ]:
def journal(events: list[tuple[str, str]]) -> Callable:
    pass

In [ ]:
def check_task_5():
    events = [("earlier", "ok")]
    decorate = journal(events)

    @decorate
    def combine(a, *, b=0):
        """Сложить два значения."""
        return a + b

    original = RuntimeError("Сбой операции")
    @decorate
    def fail():
        raise original

    assert combine(2, b=3) == 5
    assert combine(a=0) == 0
    try:
        fail()
    except RuntimeError as error:
        assert error is original
    else:
        raise AssertionError("Ожидалось исходное исключение")
    assert events == [
        ("earlier", "ok"),
        ("combine", "start"), ("combine", "ok"),
        ("combine", "start"), ("combine", "ok"),
        ("fail", "start"), ("fail", "error"),
    ]
    assert combine.__name__ == "combine"
    assert combine.__doc__ == "Сложить два значения."

    separate_events = []
    @journal(separate_events)
    def optional(*, value=None):
        return value

    assert optional() is None
    assert separate_events == [("optional", "start"), ("optional", "ok")]
    assert len(events) == 7
    print("Задание 5: проверки пройдены")

# check_task_5()

## Дополнительный практикум: стек и очередь

[Открыть семинарские материалы о стеке и очереди](../reference/stack-queue.ipynb). Там LIFO и FIFO разбираются на `list` и `collections.deque`, отмене действий, скобках и обслуживании заявок. Материал можно разобрать на отдельном семинаре; для решения пяти задач этого ноутбука он не требуется.

## Обсуждение

1. Как создаются две независимые последовательности, если код внутренней функции у них одинаковый?
2. Почему изменение списка из внешней функции обычно не требует `nonlocal`, а перепривязка числа требует?
3. Чем счётчик успешных вызовов отличается от счётчика попыток?
4. Что потеряется, если обёртка вызовет функцию, но не вернёт её результат?
5. Когда выполняется фабрика декоратора, когда сам декоратор, а когда обёртка?
6. Какие исключения уместно перехватить в журналирующем декораторе и почему после записи нужен `raise`?

## Домашняя работа

> **Домашняя работа.** В материалах курса приведены **предполагаемые списки домашних задач**. Домашние задания нужно решать и сдавать в **Яндекс.Контесте**. Актуальный список задач, условия и требования к оформлению находятся в Контесте. **При расхождениях информация в Контесте имеет приоритет.** Ссылки на контесты публикуются на **вики курса**.

Условия находятся в [tasks.md](tasks.md): три небольшие задачи и две задачи на управление состоянием обёртки.

## Итоги

Замыкание сохраняет доступ к настройке или состоянию внешнего вызова. Разные вызовы фабрики могут создать независимые экземпляры такого состояния. Обёртка добавляет поведение вокруг функции, сохраняя её договор: аргументы, результат, ошибки и сведения о самой функции.